# Caustic quickstart — a model that holds a fact and cannot reach it

A language model can carry a fact in its weights and still be unable to produce it,
and whether it can is decided by the **character** of the text in front of the
question rather than by that text's **length**. Holding a prefix at exactly 128
tokens and changing only what those tokens are, `RESULTS.md` records accuracy
**1.000** on twenty country–capital pairs when the prefix is coherent prose and
**0.000** when it is the token `" the"` repeated 128 times. The weights are
identical across those two rows. Nothing was learned or forgotten between them.

What makes this more than an anecdote is that the failure has a shape. A model that
cannot reach a fact does not answer randomly — it answers *identically*, sending
distinct entities to one shared answer. The object that records this is the **orbit
partition**: entities bucketed by the answer they receive. It consults no answer key,
and for an injective relation it certifies a lower bound on the error count by
counting alone.

This notebook runs that measurement end to end on free CPU. It runs a 135M-parameter
model first, where the headline does **not** reproduce, and then the 0.5B model the
repository's numbers were measured on, where it does. Both results are shown. The
point of an artifact like this is that a stranger can falsify the claim without
owning a GPU, which requires showing the runs that disagree as well as the ones that
agree.

In [1]:
# Install. On Colab this pulls the package straight from git; against a local clone
# of the repository it uses the checkout instead and installs nothing.
import pathlib
import sys

_local = pathlib.Path("..").resolve()  # this notebook lives in notebooks/
if (_local / "caustic" / "__init__.py").exists():
    sys.path.insert(0, str(_local))
    print(f"local clone: {_local}")
else:
    !pip install -q "caustic[experiments] @ git+https://github.com/teerthsharma/caustic.git"
    print("installed from git")

local clone: C:\Users\seal\Documents\seal\caustic\.claude\worktrees\agent-a2505a54ae2b45188


## Provenance

Everything below is deterministic: a fixed seed, `torch.no_grad()`, a model in
`.eval()`, and answers read off as the argmax next-token id rather than sampled. A
sampled answer would turn the partition into noise, so determinism is not a nicety
here — it is what makes the orbit partition an observation rather than a draw.

The versions printed by the next cell are the ones the committed outputs were
produced under. A reader whose numbers differ should compare against them first.

In [2]:
import os
import time

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"  # keep the committed outputs readable

import numpy as np
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

transformers.utils.logging.set_verbosity_error()
transformers.utils.logging.disable_progress_bar()

import caustic
from caustic import NEUTRAL_PREFIX, RelationSpec, orbit_partition, repair_by_context

SEED = 0
DEVICE = "cpu"  # forced: this notebook is written for free-tier CPU, not for CUDA
NTOK = 128  # the prefix budget, held fixed across every condition below

torch.manual_seed(SEED)

T0 = time.time()
print(f"caustic       {caustic.__version__}")
print(f"torch         {torch.__version__}")
print(f"transformers  {transformers.__version__}")
print(f"numpy         {np.__version__}")
print(f"python        {sys.version.split()[0]}")
print(f"device        {DEVICE}   seed {SEED}   prefix budget {NTOK} tokens")

caustic       0.1.0
torch         2.5.1+cu121
transformers  5.3.0
numpy         1.26.4
python        3.11.9
device        cpu   seed 0   prefix budget 128 tokens


## The relation, and the function that reads an answer out of the model

A `RelationSpec` is a prompt template plus the entities substituted into it, and a
flag recording whether the relation is **injective** — whether distinct entities
genuinely warrant distinct answers. Country and capital is injective: no two
countries here share a capital. That flag is load-bearing and not decoration. On a
many-to-one relation such as country-and-continent, several entities *should* share
an answer, a collapsed orbit is then correct behaviour, and the whole signal
inverts.

`answer_fn` maps a prompt string to a hashable answer. Here it is the argmax
next-token id. Gold answers are built alongside it but are deliberately kept out of
everything up to the certificate, so that it is visible which quantities need an
answer key and which do not.

In [3]:
PAIRS = {
    "France": "Paris", "Italy": "Rome", "Japan": "Tokyo", "Germany": "Berlin",
    "Spain": "Madrid", "Greece": "Athens", "Egypt": "Cairo", "China": "Beijing",
    "Russia": "Moscow", "Canada": "Ottawa", "Norway": "Oslo", "Sweden": "Stockholm",
    "Poland": "Warsaw", "Austria": "Vienna", "Ireland": "Dublin", "Cuba": "Havana",
    "Peru": "Lima", "Chile": "Santiago", "Thailand": "Bangkok", "Turkey": "Ankara",
}
SPEC = RelationSpec(templates=("The capital of {e} is",), entities=tuple(PAIRS))


def load(name):
    """Tokenizer and model on CPU in eval mode, gradients off."""
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32).to(DEVICE).eval()
    return tok, model


def make_answer_fn(tok, model):
    """prompt -> argmax next-token id. Deterministic by construction."""
    @torch.no_grad()
    def answer_fn(prompt: str) -> int:
        ids = torch.tensor([tok(prompt, add_special_tokens=False).input_ids], device=DEVICE)
        return int(model(ids).logits[0, -1].argmax())
    return answer_fn


def build_prefixes(tok):
    """Two prefixes of exactly NTOK tokens that differ only in character.

    Both are built as token ids and then decoded, so the budget is enforced in
    tokens rather than in characters, and the two conditions are comparable.
    """
    prose = tok.decode(tok(NEUTRAL_PREFIX * 4, add_special_tokens=False).input_ids[:NTOK])
    repeat = tok.decode((tok(" the", add_special_tokens=False).input_ids * NTOK)[:NTOK])
    n_prose = len(tok(prose, add_special_tokens=False).input_ids)
    n_repeat = len(tok(repeat, add_special_tokens=False).input_ids)
    assert n_prose == n_repeat == NTOK, f"budget not held: {n_prose} vs {n_repeat}"
    return {"coherent prose": prose, '" the" x 128': repeat}


def gold_ids(tok):
    """entity -> first token id of the correct answer. Used only after the certificate."""
    return {e: tok(" " + c, add_special_tokens=False).input_ids[0] for e, c in PAIRS.items()}


print(f"{len(PAIRS)} entities, injective={SPEC.injective}, template={SPEC.templates[0]!r}")

20 entities, injective=True, template='The capital of {e} is'


## The contrast

This is the headline, and it is a single comparison. The **token count is held
fixed at 128**. The only thing that changes between the two rows is what those 128
tokens are: coherent English on subjects unrelated to geography, against one
innocuous token repeated. If the effect were about length — position, attention
denominator, simply not being at the start of a sequence — the two rows would agree.

The first model is `HuggingFaceTB/SmolLM2-135M`, the smallest thing that can be
downloaded and run inside this notebook's budget.

In [4]:
def contrast(name):
    """Run both prefixes on one model and print the two orbit reports side by side."""
    t = time.time()
    tok, model = load(name)
    fn = make_answer_fn(tok, model)
    gold = gold_ids(tok)
    out = {}
    print(f"=== {name} ===")
    for label, prefix in build_prefixes(tok).items():
        report = orbit_partition(SPEC, lambda p, x=prefix: fn(x + p))
        acc = float(np.mean([a == gold[e] for e, a in zip(report.entities, report.answers)]))
        out[label] = (report, acc)
        print(f"\n{label:>16} | {report}")
    print(f"\n({time.time() - t:.1f}s)")
    return tok, model, fn, gold, out


tok_s, model_s, fn_s, gold_s, small = contrast("HuggingFaceTB/SmolLM2-135M")

=== HuggingFaceTB/SmolLM2-135M ===



  coherent prose | 20 entities -> 1 distinct answers, largest orbit 20  COLLAPSED: France, Italy, Japan, Germany, Spain, Greece, Egypt, China, Russia, Canada, Norway, Sweden, Poland, Austria, Ireland, Cuba, Peru, Chile, Thailand, Turkey



    " the" x 128 | 20 entities -> 2 distinct answers, largest orbit 18  COLLAPSED: France, Italy, Japan, Germany, Spain, Greece, Egypt, China, Russia, Canada, Norway, Sweden, Poland, Austria, Ireland, Peru, Chile, Turkey; Cuba, Thailand

(9.0s)


### What a 135M model actually did, which is not what the repository records

At 135M parameters the contrast does **not** reproduce. Both prefixes collapse the
partition, and coherent prose collapses it harder than the degenerate one: every
entity lands on the single token `" the"`. The prose prefix does not repair this
model, it worsens it.

The entity list and the template are the ones in `caustic/experiments/triangulate.py`
and were **not** adjusted to make this come out better. A model this small does not
reliably hold the underlying facts in the first place — with no prefix at all its
accuracy on these twenty pairs is 0.200 — and a fact the model does not hold cannot
be made unreachable, because it was never reachable. The claim under test is about a
model that *has* the fact, so 135M is below the floor where the claim is even
meaningful.

That is a limit on the demonstration, not evidence against it, and the honest way to
show it is to run the model the numbers were actually measured on immediately
afterwards, in the same notebook, under the same code.

In [5]:
tok_q, model_q, fn_q, gold_q, qwen = contrast("Qwen/Qwen2.5-0.5B")

=== Qwen/Qwen2.5-0.5B ===



  coherent prose | 20 entities -> 20 distinct answers, largest orbit 1  (no collapse)



    " the" x 128 | 20 entities -> 1 distinct answers, largest orbit 20  COLLAPSED: France, Italy, Japan, Germany, Spain, Greece, Egypt, China, Russia, Canada, Norway, Sweden, Poland, Austria, Ireland, Cuba, Peru, Chile, Thailand, Turkey

(14.6s)


## The certificate

**Theorem 1 (Orbit Error Bound).** For an injective relation on `n` entities whose
answers form a partition with `m` orbits, at least `n − m` of the answers are wrong.
The argument is counting, not statistics: injectivity means distinct entities have
distinct true answers, so if two entities share an orbit the model has given one
answer where two were owed and is wrong on at least one of them. An orbit of size
`s` therefore contributes at least `s − 1` errors, and summing over the orbits gives
`n − m`. The bound is tight, attained when every orbit contains exactly one correct
answer.

Nothing in that argument mentions a correct answer. The next cell prints the
certified floor for both prefixes **with the gold answers untouched**, and only then
reveals the measured accuracy, so the bound can be seen to hold rather than be
asserted to.

The prose row reads 0.950 here against the 1.000 recorded in `RESULTS.md`. The two
runs use different 128 tokens of prose — `coherence_vs_length.py` tokenises its own
passage, this notebook truncates `NEUTRAL_PREFIX` repeated four times — and one of
the twenty answers differs between them. The partition is identical either way, 20
orbits over 20 entities, so the certified floor is 0 in both runs.

In [6]:
print("certified WITHOUT gold answers")
print(f"{'prefix':>16} {'n':>3} {'orbits m':>9} {'largest':>8} {'certified >= n-m':>17} {'floor':>7}")
for label, (r, _) in qwen.items():
    print(f"{label:>16} {len(r.entities):>3} {r.n_distinct:>9} {r.largest_orbit:>8} "
          f"{r.certified_errors:>17} {r.certified_error_rate:>7.3f}")

print("\n...now the answer key is consulted, and only to check the bound held")
print(f"{'prefix':>16} {'accuracy':>9} {'actual wrong':>13} {'certified':>10} {'bound holds':>12}")
for label, (r, acc) in qwen.items():
    wrong = len(r.entities) - round(acc * len(r.entities))
    print(f"{label:>16} {acc:>9.3f} {wrong:>13} {r.certified_errors:>10} "
          f"{str(wrong >= r.certified_errors):>12}")

certified WITHOUT gold answers
          prefix   n  orbits m  largest  certified >= n-m   floor
  coherent prose  20        20        1                 0   0.000
    " the" x 128  20         1       20                19   0.950

...now the answer key is consulted, and only to check the bound held
          prefix  accuracy  actual wrong  certified  bound holds
  coherent prose     0.950             1          0         True
    " the" x 128     0.000            20         19         True


## The repair, which measures its own effect and can report failure

`repair_by_context` prepends `NEUTRAL_PREFIX` — coherent English about mechanical
calculators, ocean currents, language and photosynthesis, containing none of the
answers and identical across every entity — and reports the partition before and
after. It cannot be leaking answers: the same tokens in shuffled order drive
accuracy to zero. What the prefix supplies is distributional, not informational.

`repaired` is True only for a genuine collapsed-to-separated transition. `worsened`
is True when the prefix merged entities that had been separate, and it is a real
outcome rather than a defensive check — it is what fires on the 135M model below.

In [7]:
for name, fn, gold in (("SmolLM2-135M", fn_s, gold_s), ("Qwen2.5-0.5B", fn_q, gold_q)):
    rep = repair_by_context(SPEC, fn, prefix=NEUTRAL_PREFIX, gold=gold)
    print(f"{name:>14} | {rep}")
    print(f"{'':>14} | repaired={rep.repaired}  worsened={rep.worsened}  "
          f"accuracy_delta={rep.accuracy_delta:+.3f}\n")

print(f"total notebook wall clock: {time.time() - T0:.0f}s")

  SmolLM2-135M | largest orbit 15 -> 20, distinct answers 6 -> 1 of 20 entities, accuracy 0.200 -> 0.000  WORSENED
               | repaired=False  worsened=True  accuracy_delta=-0.200



  Qwen2.5-0.5B | largest orbit 4 -> 1, distinct answers 15 -> 20 of 20 entities, accuracy 0.550 -> 0.750  REPAIRED
               | repaired=True  worsened=False  accuracy_delta=+0.200

total notebook wall clock: 36s


## What this does not establish

These are the repository's own negative results, kept rather than buried.

**The detector detects collapse, not error.** Orbit collapse is one cause of wrong
answers among several. When a model is wrong through dispersed ignorance — many
distinct wrong answers, nothing colliding — the partition stays fine-grained and
nothing fires. The instrument is silent on that failure mode, and silence from it is
not evidence of correctness.

**The relation must be injective or the signal inverts.** Equivariance is undefined
on a many-to-one relation, where entities *should* share answers. Measured: **0.995
AUROC on an injective relation against 0.273 on a many-to-one one.** The 0.273 is
below chance, meaning the score is actively anti-informative there, not merely
uninformative. `RelationSpec.injective` exists to make that precondition explicit
instead of implicit.

**The headline 0.995 does not survive on harder relations.** It was measured on
`capital` and `language`. On relations chosen for low answer frequency rather than
for observed failure, pooled AUROC falls to **0.7083 [0.3809, 1.0000]** at short
context and **0.6687 [0.3819, 0.9167]** at full context. Both confidence intervals
span 0.5. On harder relations the detector **is not demonstrated to work**, and
0.995 should be read as this method's score against the one cause it was built for,
not as a hallucination-detection figure.

**And the scope of the measurements themselves.** One model family at two widths,
two injective relations of 12 and 20 entities, one distractor passage per condition,
one seed. Answers are compared by top-1 token, so a correct answer phrased
differently counts as disagreement. Whether coherence-gated retrieval is a general
property of language models or a behaviour of small models outside their training
regime is **not** established here.

## What the certificate does and does not prove

**Proves.** That at least `n − m` of these answers are wrong. This is a theorem
about counting, it holds for any injective relation, and it requires no answer key,
no held-out labels, no calibration set and no assumption about the model. When the
partition collapses to one orbit over twenty entities, nineteen answers are wrong,
and that is settled before anyone looks up a single capital city.

**Does not prove.** Which answers are wrong — the bound counts, it does not
localise. That a partition with no collapse is correct — `m = n` certifies zero
errors, which is a floor of zero and says nothing whatever about accuracy. That the
model lacks the knowledge — the whole point is that the same weights answer
correctly under a different prefix, so collapse is a statement about reachability,
not about content. And nothing at all about relations that are not injective, or
about free-form generation, where there is no set of entities and therefore no
partition to take.

Further reading: `README.md` for the abstract and the five bounds, `RESULTS.md`
section 11 for where the detector fails, and `caustic/regime.py` and
`caustic/repair.py` for the module docstrings this notebook is a walkthrough of.